In [ ]:
import pandas as pd
from report_manager import project, dataset, report, analyses, plots
import networkx as nx
from networkx.readwrite import json_graph
from plotly.offline import init_notebook_mode, iplot
import plotly.graph_objs as go
from scipy.stats import zscore
init_notebook_mode(connected=True)
%matplotlib inline
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

import warnings
warnings.filterwarnings('ignore')
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
my_project = project.Project(identifier='P0000001', datasets={}, report={})
my_project.load_project_data()
my_project.load_project_report()

In [ ]:
proteomics_dataset = my_project.get_dataset('proteomics')

In [ ]:
my_project.list_datasets()

In [ ]:
my_project.get_dataset('proteomics').list_dataframes()

In [ ]:
proteomics_dataset = my_project.get_dataset('proteomics')
processed_df = proteomics_dataset.get_dataframe('processed')

In [ ]:
processed_df.head()

In [ ]:
processed_df = processed_df.drop(['sample', 'subject'], axis=1).set_index('group').apply(zscore).reset_index()

In [ ]:
proteomics_report = my_project.get_dataset('proteomics').report

In [ ]:
# proteomics_report.list_plots()

In [ ]:
#proteomics_report.visualize_plot(environment='notebook', plot_type='13~correlation_correlation~network')

In [ ]:
correlation_net_dict = proteomics_report.get_plot('15~correlation_correlation~network')[0]

In [ ]:
correlation_net = json_graph.node_link_graph(correlation_net_dict['net_json'])

In [ ]:
correlation_df = pd.DataFrame.from_dict(correlation_net.nodes(data=True))
correlation_df = correlation_df[0].to_frame().join(correlation_df[1].apply(pd.Series))

In [ ]:
correlation_df.columns = ['identifier', 'degree', 'radius', 'color', 'cluster']

In [ ]:
min_val = processed_df._get_numeric_data().min().min().round()
max_val = processed_df._get_numeric_data().max().max().round()
processed_df = processed_df[list(correlation_df.identifier) + ['group']]

In [ ]:
from IPython.core.display import display, HTML

In [ ]:
@interact
def plot_parallel_plot(cluster=correlation_df.cluster.unique()):
    cluster_colors = dict(zip(correlation_df.cluster, correlation_df.color))
    clusters = correlation_df.groupby('cluster')
    identifiers = clusters.get_group(cluster)['identifier'].tolist()
    title= "Parallel plot cluster: {}".format(cluster)
    df = processed_df.set_index('group')[identifiers].reset_index()
    print(df.head())
    figure = plots.basicFigures.get_parallel_plot(df, identifier=cluster, args={'color':cluster_colors[cluster], 'group':'group', 
                                                                          'title':title,
                                                                          'zscore':False})
    display(HTML("<p>{}</p>".format(",".join(identifiers))))
    iplot(figure.figure)